In [38]:
import numpy as np
from PIL import Image
import pandas as pd
import json
import ast
from pathlib import Path

In [2]:
train_csv = r"E:\Dollar Street\dataset_dollarstreet\images_v2_imagenet_train.csv"
test_csv = r"E:\Dollar Street\dataset_dollarstreet\images_v2_imagenet_test.csv"

In [3]:
train_df = pd.read_csv(train_csv)
test_df = pd.read_csv(test_csv)

In [32]:
def topics2imagenet(df, map_path=r"E:\Dollar Street\dataset_dollarstreet\topics_to_imagenet_classes_map.json"):

    with open(map_path, "r") as f:
        topic_to_imagenet = json.load(f)

    def _parse_list(x): 
        if isinstance(x, list):
            return x
        if pd.isna(x):
            return []
        return ast.literal_eval(x)
    
    def _get_imagent_labels(topics):
        labels = []
        for topic in topics:
            if topic in topic_to_imagenet:
                labels.append(topic_to_imagenet[topic])
        labels = list(dict.fromkeys(labels))
        return labels
    
    df['topics'] = df['topics'] .apply(_parse_list)
    df['imagenet_labels'] = df['topics'].apply(_get_imagent_labels)
    df['label'] = df['imagenet_labels'].apply(
        lambda x: x[0] if len(x) > 0 else None
    )

    return df

In [51]:
def resize_save(df, root_path, save_path):
    for i in range(len(df)):
        image_rel_path = df.iloc[i]["imageRelPath"]
        label = df.iloc[i]["label"]
        region = df.iloc[i]["region.id"]

        if pd.isna(label) or pd.isna(region):
            continue

        input_path = root_path / image_rel_path

        if not input_path.exists():
            print("Missing:", input_path)
            continue

        img = Image.open(input_path).convert("RGB")
        img = img.resize((256, 256), Image.Resampling.LANCZOS)

        save_dir = save_path / str(region) / str(label)
        save_dir.mkdir(parents=True, exist_ok=True)

        filename = Path(image_rel_path).name
        output_path = save_dir / filename

        img.save(output_path)

In [54]:
test_df = topics2imagenet(test_df)
root_path = Path(r"E:\Dollar Street\dataset_dollarstreet")
save_path = Path(r"D:\Project\DAP_paper\datasets\DollarStreet\processed\test")
resize_save(test_df, root_path, save_path)